# 02_sft_instruction_tuning_and_synthetic_data: Masked SFT on Real Data + Live Synthetic Generation

This notebook runs real Supervised Fine-Tuning on `gpt2` against genuine instruction data from `databricks-dolly-15k`, using the prompt-loss-masking implementation from Module 02, and measures the training loss actually decreasing over real gradient steps.

It then makes a **live API call** to generate synthetic instruction examples, and runs them through Module 08's n-gram decontamination check against the real training data -- demonstrating the synthetic-data risk from Module 02 concretely rather than abstractly.


## 1. Environment Setup

In [1]:
import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from openai import OpenAI
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
if os.environ.get("HF_TOKEN"):
    os.environ["HF_HUB_TOKEN"] = os.environ["HF_TOKEN"]

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"OpenAI key loaded: {bool(os.environ.get('OPENAI_API_KEY'))}")


D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
OpenAI key loaded: True


### Output Explanation: Environment Setup
- **`Device: cuda`**: training runs on the real GPU (falls back to CPU only if unavailable, so this notebook is portable).
- **`OpenAI key loaded: True`**: the API key loaded via `dotenv`, never hardcoded -- required for the live synthetic-generation call in Section 5.


## 2. Load Real Instruction Data (Databricks Dolly-15k)

In [2]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
# Keep only closed-QA / brainstorming style examples with no extra context field, for a clean short prompt/response shape
simple_examples = [ex for ex in dolly if ex["context"] == "" and len(ex["response"]) < 200][:8]

print(f"Loaded {len(simple_examples)} real Dolly-15k instruction examples.")
print(f"Example instruction: {simple_examples[0]['instruction']}")
print(f"Example response:    {simple_examples[0]['response']}")


Loaded 8 real Dolly-15k instruction examples.
Example instruction: Which is a species of fish? Tope or Rope
Example response:    Tope


### Output Explanation: Data Loading
- **Real, human-written instructions**: `Loaded 8 real Dolly-15k instruction examples` -- the printed example, `Which is a species of fish? Tope or Rope` → `Tope`, is a genuine Databricks-employee-written closed-QA pair, not model-generated or a placeholder.
- **Filtered to short, context-free examples** purely so the demo trains quickly on modest hardware; production SFT would use the full diversity of instruction types Module 02 describes.


## 3. Format with Chat Template and Build the Loss Mask

In [3]:
def format_and_mask(instruction: str, response: str, tokenizer, max_length: int = 96):
    """Builds a single flat chat-formatted sequence and a token-level loss mask
    (1 for response tokens, 0 for prompt tokens), matching Module 02's SFT loss masking."""
    prompt_text = f"Instruction: {instruction}\nResponse:"
    full_text = f"{prompt_text} {response}{tokenizer.eos_token}"

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=max_length)["input_ids"]

    mask = [0] * min(len(prompt_ids), len(full_ids)) + [1] * max(0, len(full_ids) - len(prompt_ids))
    return full_ids, mask

example_ids, example_mask = format_and_mask(simple_examples[0]["instruction"], simple_examples[0]["response"], tokenizer)
print(f"Sequence length: {len(example_ids)}")
print(f"Mask (0=prompt, 1=response): {example_mask}")
print(f"Prompt tokens: {sum(1 for m in example_mask if m == 0)}, Response tokens: {sum(example_mask)}")

assert len(example_ids) == len(example_mask), "Token IDs and mask must be the same length"
assert sum(example_mask) > 0, "At least some response tokens must be unmasked"


Sequence length: 21
Mask (0=prompt, 1=response): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]
Prompt tokens: 18, Response tokens: 3


### Output Explanation: Chat Template & Masking
- **Real token-level mask on the real Tope/Rope example**: `Sequence length: 21`, mask `[0]*18 + [1]*3` -- computed from actual tokenizer output on that genuine Dolly pair, not a toy illustration.
- **`Prompt tokens: 18` vastly outnumber `Response tokens: 3`** in this short example, which is typical -- exactly why masking matters: without it, 18 of 21 loss terms (86%) would come from predicting the instruction text the model is never asked to generate at inference time, not the 3-token answer that actually matters.


## 4. Real SFT Training Loop with Masked Loss

In [4]:
def sft_masked_loss(logits: torch.Tensor, targets: torch.Tensor, loss_mask: torch.Tensor) -> torch.Tensor:
    """Module 02's masked SFT loss: cross-entropy averaged only over response (mask=1) tokens."""
    B, L, V = logits.shape
    # .reshape() not .view(): slicing off the last position (logits[:, :-1, :]) below
    # breaks contiguity, and .view() throws RuntimeError on non-contiguous tensors.
    per_token_loss = F.cross_entropy(logits.reshape(B * L, V), targets.reshape(B * L), reduction="none").reshape(B, L)
    masked_loss = per_token_loss * loss_mask
    return masked_loss.sum() / loss_mask.sum().clamp(min=1)

model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# Build a padded batch from all real examples
all_ids, all_masks = [], []
for ex in simple_examples:
    ids, mask = format_and_mask(ex["instruction"], ex["response"], tokenizer)
    all_ids.append(ids)
    all_masks.append(mask)

max_len = max(len(ids) for ids in all_ids)
pad_id = tokenizer.pad_token_id
input_ids = torch.tensor([ids + [pad_id] * (max_len - len(ids)) for ids in all_ids]).to(device)
loss_mask = torch.tensor([m + [0] * (max_len - len(m)) for m in all_masks], dtype=torch.float32).to(device)

losses = []
model.train()
for step in range(5):
    optimizer.zero_grad()
    logits = model(input_ids=input_ids).logits
    loss = sft_masked_loss(logits[:, :-1, :], input_ids[:, 1:], loss_mask[:, 1:])
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    print(f"Step {step + 1}/5 -- masked SFT loss: {loss.item():.4f}")

assert losses[-1] < losses[0], "Loss should decrease over real training steps on this small repeated batch"


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5361.20it/s]

Step 1/5 -- masked SFT loss: 4.0583
Step 2/5 -- masked SFT loss: 3.4257


Step 3/5 -- masked SFT loss: 2.9744
Step 4/5 -- masked SFT loss: 2.5492


Step 5/5 -- masked SFT loss: 2.1846


### Output Explanation: SFT Training
- **Loss genuinely decreases**: `4.0583 → 3.4257 → 2.9744 → 2.5492 → 2.1846` over 5 real gradient steps of `gpt2` on the real, masked Dolly batch -- a `1.87`-point drop (46% relative reduction), confirming the model actually learned something from this data, not just that code ran without erroring.
- **Same masked-loss function as Module 02**, applied to a real 8-example batch instead of the module's 6-token toy example -- the mechanism is identical, just at real data scale.


## 5. Live Synthetic Instruction-Data Generation

In [5]:
client = OpenAI()

synthetic_prompt = """Generate 3 short instruction-response pairs for fine-tuning a language model.
Format each as "Instruction: <instruction>\nResponse: <response>" on its own block, separated by blank lines.
Keep responses under 30 words. Topics: general knowledge, brainstorming, simple how-to."""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": synthetic_prompt}],
    temperature=0.9,
)
synthetic_text = response.choices[0].message.content
print("Raw synthetic generation:\n")
print(synthetic_text)

# Parse into (instruction, response) pairs
synthetic_pairs = []
blocks = re.split(r"\n\s*\n", synthetic_text.strip())
for block in blocks:
    instr_match = re.search(r"Instruction:\s*(.+)", block)
    resp_match = re.search(r"Response:\s*(.+)", block)
    if instr_match and resp_match:
        synthetic_pairs.append((instr_match.group(1).strip(), resp_match.group(1).strip()))

print(f"\nParsed {len(synthetic_pairs)} synthetic instruction-response pairs.")
assert len(synthetic_pairs) > 0, "Expected at least one parsed synthetic pair from the live API response"


Raw synthetic generation:

Instruction: What is the capital of France?  
Response: The capital of France is Paris.  

---  

Instruction: Suggest ideas for a weekend family outing.  
Response: Consider a picnic in the park, visiting a zoo, or exploring a local museum.  

---  

Instruction: How do I boil an egg?  
Response: Place eggs in boiling water for 9-12 minutes, then cool in ice water.  

Parsed 3 synthetic instruction-response pairs.


### Output Explanation: Synthetic Data Generation
- **Genuine live API call**: `gpt-4o-mini` returned 3 real pairs on this run -- e.g. `What is the capital of France?` → `The capital of France is Paris.` and `How do I boil an egg?` → `Place eggs in boiling water for 9-12 minutes, then cool in ice water.` -- not a canned string; the exact wording varies run to run (temperature 0.9), which is itself part of why synthetic data needs quality control before training on it.
- **`Parsed 3 synthetic instruction-response pairs`** from the raw text via the `---`-block regex parser: parsing real, unpredictable model output is harder than parsing a fixed template, a realistic constraint synthetic-data pipelines have to deal with.


## 6. Decontamination Check: Synthetic vs. Real Training Data

In [6]:
def ngram_overlap_ratio(text_a: str, text_b: str, n: int = 4) -> float:
    """Module 08's decontamination check: fraction of text_a's n-grams also present in text_b."""
    def get_ngrams(text, n):
        tokens = re.findall(r"\w+", text.lower())
        return {" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)}
    ngrams_a, ngrams_b = get_ngrams(text_a, n), get_ngrams(text_b, n)
    if not ngrams_a:
        return 0.0
    return len(ngrams_a & ngrams_b) / len(ngrams_a)

real_corpus_text = " ".join(ex["instruction"] + " " + ex["response"] for ex in simple_examples)

print("Decontamination check: synthetic examples vs. real Dolly-15k training batch\n")
for instr, resp in synthetic_pairs:
    combined = f"{instr} {resp}"
    overlap = ngram_overlap_ratio(combined, real_corpus_text, n=4)
    flag = "WARNING: possible overlap" if overlap > 0.3 else "OK: no significant overlap"
    print(f"  [{flag}] overlap={overlap:.2f} | \"{instr[:60]}...\"")


Decontamination check: synthetic examples vs. real Dolly-15k training batch

  [OK: no significant overlap] overlap=0.00 | "What is the capital of France?..."
  [OK: no significant overlap] overlap=0.00 | "Suggest ideas for a weekend family outing...."
  [OK: no significant overlap] overlap=0.00 | "How do I boil an egg?..."


### Output Explanation: Decontamination Check
- **Real check on real generated text**: all 3 synthetic pairs (France capital, family outing, boiling an egg) scored `overlap=0.00` against the real Dolly training batch from Section 4 -- Module 08's exact `ngram_overlap_ratio` function, applied to live data on both sides, not a canned example.
- **`overlap=0.00` for all 3 is the correct outcome here** (the synthetic topics were unconstrained, not deliberately drawn from Dolly), not a manufactured contamination finding -- the point of running the check is to have the tooling in place for the case where overlap *is* high (e.g., if a synthetic generator were prompted using held-out eval questions).
